#Setting Up the Models

##Install Libraries and Import Packages

In [ ]:
!pip install -r requirements_pm.txt

In [ ]:
from medmnist import PathMNIST

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import keras.utils
from keras.utils import to_categorical

from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
import tensorflow as tf
from tensorflow.keras import Input, Model

from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix, accuracy_score, auc
import matplotlib.pyplot as plt
import seaborn as sns
import cv2

import shap
import sage

from scipy.stats import spearmanr
from collections import Counter

import random

import warnings
warnings.filterwarnings('ignore')

state = 100
SEED = 100
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

##Downloading Training and Testing Data, then Spliting them

In [ ]:
train_dataset = PathMNIST(split='train', download=True)
test_dataset  = PathMNIST(split='test', download=True)

X = np.concatenate([train_dataset.imgs, test_dataset.imgs], axis=0)
y = np.concatenate([train_dataset.labels, test_dataset.labels], axis=0).ravel()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=100, stratify=y)

In [ ]:
train_dataset.imgs[100]

##Z Score Functions for the Images

In [ ]:
def zscore(images):
    imgs = images.astype(np.float32)
    mean = np.mean(imgs)
    std  = np.std(imgs) + 1e-7
    return (imgs - mean) / std

In [ ]:
def zscore_cnn(images):
    images = images.astype(np.float32)
    mean = np.mean(images, axis=(1,2,3), keepdims=True)
    std = np.std(images, axis=(1,2,3), keepdims=True) + 1e-7
    return (images - mean) / std

##Data Processing

In [ ]:
X_train_norm = zscore(X_train)
X_test_norm = zscore(X_test)

In [ ]:
X_train_cnn = zscore_cnn(X_train)
X_test_cnn = zscore_cnn(X_test)

In [ ]:
num_classes = 9

y_train_cat = to_categorical(y_train, num_classes)
y_test_cat = to_categorical(y_test, num_classes)

In [ ]:
X_train_flat = X_train_norm.reshape(X_train_norm.shape[0], -1)
X_test_flat = X_test_norm.reshape(X_test_norm.shape[0], -1)

##Creating the Optimized Models and Then Fitting them

In [ ]:
dt = DecisionTreeClassifier(ccp_alpha=0.0, class_weight=None, criterion='entropy',
                            max_depth=9, max_features=None, max_leaf_nodes=None,
                            min_impurity_decrease=0.0, min_samples_leaf=17, min_samples_split=9,
                            min_weight_fraction_leaf=0.0, monotonic_cst=None, random_state=100,
                            splitter='best')

lr = LogisticRegression(C=0.023925275054862166, class_weight='balanced', dual=False,
                        fit_intercept=True, intercept_scaling=1, l1_ratio=None,
                        max_iter=580, multi_class='deprecated', n_jobs=None,
                        penalty='l1', random_state=100, solver='liblinear',
                        tol=0.0001, verbose=0, warm_start=False)

xgb = XGBClassifier(objective='multi:softprob', base_score=None, booster=None,
                    callbacks=None, colsample_bylevel=None, colsample_bynode=None,
                    colsample_bytree=0.8088882995005343, device=None, early_stopping_rounds=None,
                    enable_categorical=False, eval_metric=None, feature_types=None,
                    feature_weights=None, gamma=0.6498114080660078, grow_policy=None,
                    importance_type=None, interaction_constraints=None, learning_rate=0.10038925516819777,
                    max_bin=None, max_cat_threshold=None, max_cat_to_onehot=None,
                    max_delta_step=None, max_depth=6, max_leaves=None,
                    min_child_weight=8, missing=np.nan, monotone_constraints=None,
                    multi_strategy=None, n_estimators=600, n_jobs=-1,
                    num_parallel_tree=None, random_state=100, reg_alpha=7.132786759581785,
                    reg_lambda=9.85141794610253, sampling_method=None, scale_pos_weight=None,
                    subsample=0.7868998413877891, tree_method=None, validate_parameters=None,
                    verbosity=None)

cnn = tf.keras.models.load_model("CNN_final.keras")
cnn.predict(np.zeros((1, 28, 28, 3), dtype=np.float32))

In [ ]:
dt.fit(X_train_flat, y_train)
lr.fit(X_train_flat, y_train)
xgb.fit(X_train_flat, y_train)

In [ ]:
cnn.summary()

##Evaluating Each Model's Performance

In [ ]:
dt_pred = dt.predict(X_test_flat)
dt_proba = dt.predict_proba(X_test_flat)
print("\nFor the Decision Tree Model:\nAccuracy: " + str(accuracy_score(y_test, dt_pred) * 100) + "%")
print("ROC AUC score: " + str(roc_auc_score(y_test, dt_proba, multi_class='ovr')))
print("Classification Report:\n" + str(classification_report(y_test, dt_pred)))
print("Confusion Matrix\n" + str(confusion_matrix(y_test, dt_pred)))

In [ ]:
lr_pred = lr.predict(X_test_flat)
lr_proba = lr.predict_proba(X_test_flat)
print("\nFor the Logistic Regression Model:\nAccuracy: " + str(accuracy_score(y_test, lr_pred) * 100) + "%")
print("ROC AUC score: " + str(roc_auc_score(y_test, lr_proba, multi_class='ovr')))
print("Classification Report:\n" + str(classification_report(y_test, lr_pred)))
print("Confusion Matrix\n" + str(confusion_matrix(y_test, lr_pred)))

In [ ]:
xgb_pred = xgb.predict(X_test_flat)
xgb_proba = xgb.predict_proba(X_test_flat)
print("\nFor the XGBoost Model:\nAccuracy: " + str(accuracy_score(y_test, xgb_pred) * 100) + "%")
print("ROC AUC score: " + str(roc_auc_score(y_test, xgb_proba, multi_class='ovr')))
print("Classification Report:\n" + str(classification_report(y_test, xgb_pred)))
print("Confusion Matrix\n" + str(confusion_matrix(y_test, xgb_pred)))

In [ ]:
cnn_proba = cnn.predict(X_test_cnn)
cnn_pred = np.argmax(cnn_proba, axis=1)
print("\nFor the CNN Model:\nAccuracy: " + str(accuracy_score(y_test, cnn_pred)*100) + "%")
print("ROC AUC score: " + str(roc_auc_score(y_test_cat, cnn_proba, multi_class='ovr')))
print("Classification Report:\n" + str(classification_report(y_test, cnn_pred)))
print("Confusion Matrix\n" + str(confusion_matrix(y_test, cnn_pred)))

##Setting Up SHAP explainers

In [ ]:
rng = np.random.RandomState(SEED)
background_idx = rng.choice(X_train_flat.shape[0], 256, replace=False)
background_flat = X_train_flat[background_idx]
background_imgs = X_train_cnn[background_idx]

n_samples = 512

explain_flat = X_test_flat[:n_samples]
explain_imgs = X_test_cnn[:n_samples]

print("Background shape (flat): " + str(background_flat.shape))
print("Explain samples shape (flat): " + str(explain_flat.shape) + "\n")

In [ ]:
dt_explainer = shap.TreeExplainer(dt)
dt_shap_values = dt_explainer.shap_values(explain_flat)
print("DT SHAP shape: " + str(np.array(dt_shap_values).shape) + "\n")

lr_explainer = shap.LinearExplainer(lr, background_flat)
lr_shap_values = lr_explainer.shap_values(explain_flat)
print("LR SHAP shape: " + str(np.array(lr_shap_values).shape) + "\n")

xgb_explainer = shap.TreeExplainer(xgb, feature_perturbation='interventional')
xgb_shap_values = xgb_explainer.shap_values(explain_flat)
print("XGB SHAP shape: " + str(np.array(xgb_shap_values).shape) + "\n")

cnn_explainer = shap.DeepExplainer(cnn, background_imgs)
cnn_shap_values = cnn_explainer.shap_values(explain_imgs)
print("CNN SHAP shape: " + str(np.array(cnn_shap_values).shape))

cnn_shap_flat = cnn_shap_values.reshape(cnn_shap_values.shape[0], -1, cnn_shap_values.shape[4])
print("New CNN SHAP shape: " + str(np.array(cnn_shap_flat).shape))

#SHAP Evaluation

##Model Agnostic XAI Methods For the CNN Model

In [ ]:
def build_gradcam_model(model, target_layer_name):
    input_tensor = Input(shape=model.input_shape[1:])
    x = input_tensor
    conv_output = None

    for layer in model.layers:
        x = layer(x)
        if layer.name == target_layer_name:
            conv_output = x

    final_output = x
    grad_model = Model(inputs=input_tensor, outputs=[conv_output, final_output])

    return grad_model, target_layer_name

def make_gradcam_heatmap(img, grad_model):
    img_array = np.expand_dims(img, axis=0).astype(np.float32)

    with tf.GradientTape() as tape:
        conv_outputs, preds = grad_model(img_array, training=False)
        pred_index = tf.argmax(preds[0])
        class_channel = preds[:, pred_index]

    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0)
    heatmap = heatmap / (tf.reduce_max(heatmap) + 1e-10)

    heatmap_resized = cv2.resize(heatmap.numpy(), (img.shape[1], img.shape[0]))

    return heatmap_resized

In [ ]:
def get_gradients(img_input, model, top_pred_idx):
    images = tf.cast(img_input, tf.float32)

    with tf.GradientTape() as tape:
        tape.watch(images)
        preds = model(images)
        top_class = preds[:, top_pred_idx]

    grads = tape.gradient(top_class, images)
    return grads

def get_integrated_gradients(img_input, model, num_steps=50):
    preds = model.predict(np.expand_dims(img_input, axis=0), verbose=0)
    top_pred_idx = np.argmax(preds[0])

    baseline = np.zeros_like(img_input).astype(np.float32)
    img_input = img_input.astype(np.float32)

    interpolated_images = [baseline + (step / num_steps) * (img_input - baseline)
                           for step in range(num_steps + 1)]
    interpolated_images = np.array(interpolated_images).astype(np.float32)

    grads = []
    batch_size = 10

    for i in range(0, len(interpolated_images), batch_size):
        batch = interpolated_images[i:i+batch_size]
        batch_grads = get_gradients(batch, model, top_pred_idx)
        grads.append(batch_grads)

    grads = np.concatenate(grads, axis=0)

    grads = (grads[:-1] + grads[1:]) / 2.0
    avg_grads = np.mean(grads, axis=0)

    integrated_grads = (img_input - baseline) * avg_grads

    return integrated_grads

In [ ]:
grad_model, layer_used = build_gradcam_model(cnn, target_layer_name="conv2d_2")

gradcam_heatmaps = []
ig_attributions = []

for i in range(n_samples):
    heatmap = make_gradcam_heatmap(explain_imgs[i], grad_model)
    heatmap_3d = np.stack([heatmap] * 3, axis=-1)
    heatmap_flat = heatmap_3d.flatten()
    heatmap_flat = (heatmap_flat - heatmap_flat.min()) / (heatmap_flat.max() + 1e-10)
    gradcam_heatmaps.append(heatmap_flat)

    ig_attr = get_integrated_gradients(explain_imgs[i], cnn)
    ig_flat = np.abs(ig_attr).flatten()
    ig_flat = (ig_flat - ig_flat.min()) / (ig_flat.max() + 1e-10)
    ig_attributions.append(ig_flat)

gradcam_heatmaps = np.array(gradcam_heatmaps)
ig_attributions = np.array(ig_attributions)

print("Grad-CAM Heatmap shape: " + str(gradcam_heatmaps.shape))
print("Integrated Gradients Attributions shape: " + str(ig_attributions.shape))

gradcam_importances = np.mean(gradcam_heatmaps, axis=0)
ig_importances = np.mean(ig_attributions, axis=0)

print("Grad-CAM Importances shape: " + str(gradcam_importances.shape))
print("Integrated Gradients Importances shape: " + str(ig_importances.shape))

In [ ]:
print("Decision Tree Importances shape: " + str(dt.feature_importances_.shape))
print("Logistic Regression Importances shape: " + str(np.abs(lr.coef_).mean(axis=0).shape))
print("XGBoost Importances shape: " + str(xgb.feature_importances_.shape))

In [ ]:
models_dict = {
    'DT': (dt, dt_explainer, dt_shap_values, dt.feature_importances_, False),
    'LR': (lr, lr_explainer, lr_shap_values, np.abs(lr.coef_).mean(axis=0), False),
    'XGB': (xgb, xgb_explainer, xgb_shap_values, xgb.feature_importances_, False),
    'CNN_GradCAM': (cnn, cnn_explainer, cnn_shap_flat, gradcam_importances, True),
    'CNN_IG': (cnn, cnn_explainer, cnn_shap_flat, ig_importances, True)
}

##Fidelity (Correlation between model and SHAP feature importances)

In [ ]:
def fidelity(model_importance, shap_values):
    shap_importance = np.abs(shap_values).mean(axis=(0, 2))
    return spearmanr(model_importance, shap_importance)[0]

##Consistency (Entropy of top feature across instances)

In [ ]:
def shap_consistency(shap_values, n_features=2352):
    sample_importance = np.abs(shap_values).mean(axis=2)
    top_feature = np.argmax(sample_importance, axis=1)

    value, counts = np.unique(top_feature, return_counts=True)
    probs = counts / len(top_feature)
    entropy = -np.sum(probs * np.log(probs + 1e-10))

    dominant_feature = f"pixel_{value[np.argmax(counts)]}"
    dominant_percent = counts.max() / len(top_feature)

    return entropy, dominant_feature, dominant_percent

##Concentration (Entropy of explanation values)

In [ ]:
def shap_concentration(shap_values, n_features=2352):
    if shap_values.ndim == 5:
            shap_values = shap_values.reshape(shap_values.shape[0], -1, shap_values.shape[-1])

    vals = np.abs(shap_values).mean(axis=(0, 2))

    total = vals.sum()
    probs = vals / (total + 1e-12)

    entropy = -np.sum(probs * np.log(probs + 1e-12))

    dominant_idx = np.argmax(vals)
    dominant_feature = f"pixel_{dominant_idx}"
    dominant_percent = vals[dominant_idx] / total

    return entropy, dominant_feature, dominant_percent

##Robustness (Test if SHAP values remain stable under small perturbations)

In [ ]:
def local_robustness(explainer, X_sample, n_instances=10, n_perturbations=10, noise_std=0.1, seed=SEED, is_cnn=False):
    rng = np.random.RandomState(seed)
    stabilities = []

    for i in range(min(n_instances, len(X_sample))):
        instance = X_sample[i:i+1]

        if is_cnn:
            base_shap = explainer.shap_values(instance, check_additivity=False)
            base_shap = base_shap.reshape(base_shap.shape[0], -1, base_shap.shape[-1])
        else:
            base_shap = explainer.shap_values(instance)

        base_shap_avg = np.abs(base_shap).mean(axis=(0, 2))

        corrs = []
        for _ in range(n_perturbations):
            noise = rng.normal(0, noise_std, instance.shape)
            perturbed = instance + noise

            if is_cnn:
                perturbed_shap = explainer.shap_values(perturbed, check_additivity=False)
                perturbed_shap = perturbed_shap.reshape(perturbed_shap.shape[0], -1, perturbed_shap.shape[-1])
            else:
               perturbed_shap = explainer.shap_values(perturbed)

            perturbed_shap_avg = np.abs(perturbed_shap).mean(axis=(0, 2))
            corr = spearmanr(base_shap_avg, perturbed_shap_avg)[0]
            corrs.append(corr)

        stabilities.append(np.mean(corrs))

    return np.mean(stabilities) if stabilities else 0.0


In [ ]:
def global_robustness(model, background, X_test, n_perturbations=10, noise_std=0.1, seed=100, is_cnn=False):
    rng = np.random.RandomState(seed)

    if model == dt:
        explainer = shap.TreeExplainer(dt)
    elif model == lr:
        explainer = shap.LinearExplainer(lr, background)
    elif model == xgb:
        explainer = shap.TreeExplainer(xgb, feature_perturbation='interventional')
    elif model == cnn:
        explainer = shap.DeepExplainer(cnn, background)

    if is_cnn:
        base_shap = explainer.shap_values(X_test, check_additivity=False)
        base_shap = base_shap.reshape(base_shap.shape[0], -1, base_shap.shape[4])
    else:
        base_shap = explainer.shap_values(X_test)

    base_global = np.abs(base_shap).mean(axis=(0, 2))

    correlations = []

    for _ in range(n_perturbations):
        noise = rng.normal(0, noise_std, X_test.shape)
        X_test_perturbed = X_test + noise

        if is_cnn:
            pert_shap = explainer.shap_values(X_test_perturbed, check_additivity=False)
            pert_shap = pert_shap.reshape(pert_shap.shape[0], -1, pert_shap.shape[4])
        else:
            pert_shap = explainer.shap_values(X_test_perturbed)

        pert_global = np.abs(pert_shap).mean(axis=(0, 2))
        correlations.append(spearmanr(base_global, pert_global)[0])

    return float(np.mean(correlations))

##Insertion AUC

In [ ]:
def insertion_auc(model, X_test_flat, shap_values, n_samples=100, steps=50, is_CNN=False):
    aucs = []
    n_samples = min(n_samples, len(X_test_flat))

    for i in range(n_samples):
        img_flat = X_test_flat[i]
        img = img_flat.reshape(28, 28, 3)

        shap_importance = np.abs(shap_values[i]).mean(axis=1)

        shap_spatial = shap_importance.reshape(28, 28, 3)
        attr = shap_spatial.mean(axis=-1)

        H, W, C = img.shape

        baseline = cv2.GaussianBlur(img, (11, 11), 10)

        if is_CNN:
            pred = model.predict(img[np.newaxis], verbose=0)
        else:
            pred = model.predict_proba(img_flat.reshape(1, -1))
        pred_class = np.argmax(pred[0])

        pixel_order = np.argsort(attr.flatten())[::-1]

        current_img = baseline.copy()
        confidences = []

        total = H * W
        pixels_per_step = max(1, total // steps)

        for step in range(steps):
            start_idx = step * pixels_per_step
            end_idx = (step + 1) * pixels_per_step if step < steps - 1 else total

            for pixel_idx in pixel_order[start_idx:end_idx]:
                row, col = pixel_idx // W, pixel_idx % W
                current_img[row, col] = img[row, col]

            if is_CNN:
                pred = model.predict(current_img[np.newaxis], verbose=0)
            else:
                pred = model.predict_proba(current_img.reshape(1, -1))

            confidences.append(pred[0][pred_class])

        x = np.linspace(0, 1, len(confidences))
        auc_score = auc(x, confidences)
        aucs.append(auc_score)

    return np.mean(aucs), np.std(aucs)

##Deletion AUC

In [ ]:
def deletion_auc(model, X_test_flat, shap_values, n_samples=100, steps=50, is_CNN=False):
    aucs = []
    n_samples = min(n_samples, len(X_test_flat))

    for i in range(n_samples):
        img_flat = X_test_flat[i]
        img = img_flat.reshape(28, 28, 3)

        shap_importance = np.abs(shap_values[i]).mean(axis=1)
        shap_spatial = shap_importance.reshape(28, 28, 3)
        attr = shap_spatial.mean(axis=-1)

        H, W, C = img.shape

        if is_CNN:
            pred = model.predict(img[np.newaxis], verbose=0)
        else:
            pred = model.predict_proba(img_flat.reshape(1, -1))
        pred_class = np.argmax(pred[0])

        pixel_order = np.argsort(attr.flatten())[::-1]

        current_img = img.copy()
        mean_color = img.mean(axis=(0, 1))

        confidences = []

        total = H * W
        pixels_per_step = max(1, total // steps)

        for step in range(steps):
            start_idx = step * pixels_per_step
            end_idx = (step + 1) * pixels_per_step if step < steps - 1 else total

            for pixel_idx in pixel_order[start_idx:end_idx]:
                row, col = pixel_idx // W, pixel_idx % W
                current_img[row, col] = mean_color

            if is_CNN:
                pred = model.predict(current_img[np.newaxis], verbose=0)
            else:
                pred = model.predict_proba(current_img.reshape(1, -1))

            confidences.append(pred[0][pred_class])

        x = np.linspace(0, 1, len(confidences))
        auc_score = auc(x, confidences)
        aucs.append(auc_score)

    return np.mean(aucs), np.std(aucs)

##SHAP Evaluation Results

In [ ]:
results = {}

for model_name, (model, explainer, shap_vals, feat_imp, is_cnn) in models_dict.items():
    fl_score = fidelity(feat_imp, shap_vals)

    n_features = 2352
    shap_consistency_entropy, shap_consistency_dominant_feature, shap_consistency_dominant_percent = shap_consistency(shap_vals)
    shap_consistency_score = 1 - (shap_consistency_entropy / np.log(n_features))#min-max normalization

    shap_concentration_entropy, shap_concentration_dominant_feature, shap_concentration_dominant_percent = shap_concentration(shap_vals)
    shap_concentration_score = 1 - (shap_concentration_entropy / np.log(n_features))

    if is_cnn:
        X_local = explain_imgs[:20]
        X_global = explain_imgs[:256]
        background = background_imgs
    else:
        X_local = explain_flat[:20]
        X_global = explain_flat[:256]
        background = background_flat

    local_rb_baseline = local_robustness(explainer, X_local, seed=SEED, is_cnn=is_cnn)
    local_rb_more_perturbations = local_robustness(explainer, X_local, n_perturbations=30, seed=SEED, is_cnn=is_cnn)
    local_rb_higher_noise = local_robustness(explainer, X_local, noise_std=0.3, seed=SEED, is_cnn=is_cnn)

    global_rb_baseline = global_robustness(model, background, X_global, seed=SEED, is_cnn=is_cnn)
    global_rb_more_perturbations = global_robustness(model, background, X_global, n_perturbations=30, seed=SEED, is_cnn=is_cnn)
    global_rb_higher_noise = global_robustness(model, background, X_global, noise_std=0.3, seed=SEED, is_cnn=is_cnn)

    ins_mean, ins_std = insertion_auc(model, explain_flat, shap_vals, is_CNN=is_cnn)
    del_mean, del_std = deletion_auc(model, explain_flat, shap_vals, is_CNN=is_cnn)


    results[model_name] = {
        'fidelity': fl_score,
        'shap_consistency': shap_consistency_score,
        'shap_consistency_dominant_feature': shap_consistency_dominant_feature,
        'shap_consistency_dominant_percent': shap_consistency_dominant_percent,
        'shap_concentration': shap_concentration_score,
        'shap_concentration_dominant_feature': shap_concentration_dominant_feature,
        'shap_concentration_dominant_percent': shap_concentration_dominant_percent,
        'local_robustness_baseline': local_rb_baseline,
        'local_robustness_more_perturbations': local_rb_more_perturbations,
        'local_robustness_higher_noise': local_rb_higher_noise,
        'global_robustness_baseline': global_rb_baseline,
        'global_robustness_more_perturbations': global_rb_more_perturbations,
        'global_robustness_higher_noise': global_rb_higher_noise,
        'insertion_auc_mean': ins_mean,
        'insertion_auc_std': ins_std,
        'deletion_auc_mean': del_mean,
        'deletion_auc_std': del_std
    }

##Results per Model

In [ ]:
model_names = ['DT', 'LR', 'XGB', 'CNN_GradCAM', 'CNN_IG']

##Fidelity Plot

In [ ]:
fl_scores = [results[m]['fidelity'] for m in model_names]
plt.bar(model_names, fl_scores)
plt.ylabel('Correlation')
plt.title('SHAP Fidelity')
plt.show()

In [ ]:
for model in model_names:
    print("Model: " + str(model))
    print("fidelity: " + str(results[model]['fidelity']))
    print("")

In [ ]:
gradcam_ig_agreement = spearmanr(gradcam_importances, ig_importances)[0]
print("Grad-CAM and Integrated Gradients Agreement: " + str(gradcam_ig_agreement))

##Consistency Plot

In [ ]:
shap_consistency_scores = [results[m]['shap_consistency'] for m in model_names]
plt.bar(model_names, shap_consistency_scores)
plt.ylabel('Correlation')
plt.title('SHAP Consistency Scores')
plt.show()

In [ ]:
for model in model_names:
    print("Model: " + str(model))
    for metric in ['shap_consistency_dominant_feature', 'shap_consistency_dominant_percent']:
        print(str(metric) + ": " + str(results[model][metric]))
    print("")

##Concentration Plot

In [ ]:
shap_concentration_scores = [results[m]['shap_concentration'] for m in model_names]
plt.bar(model_names, shap_concentration_scores)
plt.ylabel('Correlation')
plt.title('Shap Concentration Scores')
plt.show()

In [ ]:
for model in model_names:
    print("Model: " + str(model))
    for metric in ['shap_concentration_dominant_feature', 'shap_concentration_dominant_percent']:
        print(str(metric) + ": " + str(results[model][metric]))
    print("")

##Robustness Testing Results for each Model

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(model_names))
width = 0.2

rects1 = ax.bar(x - 1.5*width, [results[m]['local_robustness_baseline'] for m in model_names], width, label='RB 1')
rects2 = ax.bar(x - 0.5*width, [results[m]['local_robustness_more_perturbations'] for m in model_names], width, label='RB 2')
rects3 = ax.bar(x + 0.5*width, [results[m]['local_robustness_higher_noise'] for m in model_names], width, label='RB 3')

ax.set_ylabel('Correlation')
ax.set_title('Local SHAP Robustness by Model')
ax.set_xticks(x)
ax.set_xticklabels(model_names)
ax.legend(['RB 1 (n_perturbations=10, noise_std=0.1)', 'RB 2 (n_perturbations=30, noise_std=0.1)', 'RB 3 (n_perturbations=10, noise_std=0.3)'])

plt.tight_layout()
plt.show()

In [ ]:
for model in model_names:
    print("Model: " + str(model))
    for metric in ['local_robustness_baseline', 'local_robustness_more_perturbations', 'local_robustness_higher_noise']:
        print(str(metric) + ": " + str(results[model][metric]))
    print("")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(model_names))
width = 0.2

rects1 = ax.bar(x - 1.5*width, [results[m]['global_robustness_baseline'] for m in model_names], width, label='RB 1')
rects2 = ax.bar(x - 0.5*width, [results[m]['global_robustness_more_perturbations'] for m in model_names], width, label='RB 2')
rects3 = ax.bar(x + 0.5*width, [results[m]['global_robustness_higher_noise'] for m in model_names], width, label='RB 3')

ax.set_ylabel('Correlation')
ax.set_title('Global SHAP Robustness by Model')
ax.set_xticks(x)
ax.set_xticklabels(model_names)
ax.legend(['RB 1 (n_perturbations=10, noise_std=0.1)', 'RB 2 (n_perturbations=30, noise_std=0.1)', 'RB 3 (n_perturbations=10, noise_std=0.3)'])

plt.tight_layout()
plt.show()

In [ ]:
for model in model_names:
    print("Model: " + str(model))
    for metric in ['global_robustness_baseline', 'global_robustness_more_perturbations', 'global_robustness_higher_noise']:
        print(str(metric) + ": " + str(results[model][metric]))
    print("")

##Insertion Plots

In [ ]:
ins_mean_scores = [results[m]['insertion_auc_mean'] for m in model_names]
plt.bar(model_names, ins_mean_scores)
plt.title('Insetion Auc Mean')
plt.show()

In [ ]:
ins_std_scores = [results[m]['insertion_auc_std'] for m in model_names]
plt.bar(model_names, ins_std_scores)
plt.title('Insetion Auc Std')
plt.show()

In [ ]:
for model in model_names:
    print("Model: " + str(model))
    for metric in ['insertion_auc_mean', 'insertion_auc_std']:
        print(str(metric) + ": " + str(results[model][metric]))
    print("")

##Deletion Plot

In [ ]:
del_mean_scores = [results[m]['deletion_auc_mean'] for m in model_names]
plt.bar(model_names, del_mean_scores)
plt.title('Deletion Auc Mean')
plt.show()

In [ ]:
del_std_scores = [results[m]['deletion_auc_std'] for m in model_names]
plt.bar(model_names, del_std_scores)
plt.title('Deletion Auc Std')
plt.show()

In [ ]:
for model in model_names:
    print("Model: " + str(model))
    for metric in ['deletion_auc_mean', 'deletion_auc_std']:
        print(str(metric) + ": " + str(results[model][metric]))
    print("")

##Accuracy vs Explainability Plot (Through mean values of metrics)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

explain_scores = {
    m: np.mean([
        results[m]['fidelity'],
        results[m]['shap_consistency'],
        results[m]['shap_concentration'],
        results[m]['local_robustness_baseline'],
        results[m]['global_robustness_baseline'],
        results[m]['insertion_auc_mean'],
        1-results[m]['deletion_auc_mean']
    ]) for m in model_names
}

accuracies = {
    'DT': accuracy_score(y_test, dt_pred),
    'LR': accuracy_score(y_test, lr_pred),
    'XGB': accuracy_score(y_test, xgb_pred),
    'CNN_GradCAM': accuracy_score(y_test, cnn_pred),
    'CNN_IG': accuracy_score(y_test, cnn_pred)
}

for model in model_names:
    ax.scatter(explain_scores[model], accuracies[model])
    ax.annotate(model, (explain_scores[model], accuracies[model]))

ax.set_xlabel('Explainability Score (0-1)')
ax.set_ylabel('Model Accuracy')
ax.set_title('Accuracy vs Explainability Tradeoff')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
for model in model_names:
    print("Model: " + str(model) + ", Accuracy: " + str(accuracies[model]) + ", Explainability Score:" + str(explain_scores[model]))